In [46]:
import pandas as pd
import numpy as np
import re
import random

from tqdm import tqdm
tqdm.pandas()

In [47]:
df=pd.read_csv(r"C:\Users\hp\Downloads\UNI\Spring 2026\Bachelor\Bachelor_Project\data\combined_processed_v1.csv")

In [48]:
# =====================================
# IMPLICIT HATE TEMPLATE AUGMENTATION
# =====================================

target_groups = [

    "muslims",
    "christians",
    "jews",
    "women",
    "immigrants",
    "black people",
    "asians",
    "disabled people",
    "gay people",
    "foreigners"

]

hate_templates = [

    "i hate {}",
    "{} are disgusting",
    "{} are ruining society",
    "{} should leave this country",
    "{} are inferior",
    "{} are dangerous",
    "{} should disappear",
    "we do not want {} here",
    "{} are criminals",
    "{} are the problem",
    "i cannot stand {}",
    "{} make society worse",
    "{} should not exist",
    "all {} are bad",
    "{} are useless"

]

implicit_hate_samples = []

for group in target_groups:

    for template in hate_templates:

        text = template.format(group)

        implicit_hate_samples.append(
            (text, 1)
        )

print("Generated samples:", len(implicit_hate_samples))

Generated samples: 150


In [49]:
implicit_df = pd.DataFrame(
    implicit_hate_samples,
    columns=["clean_text", "class"]
)

implicit_df.head()

,clean_text,class
0,i hate muslims,1
1,muslims are disgusting,1
2,muslims are ruining society,1
3,muslims should leave this country,1
4,muslims are inferior,1


In [50]:
df = pd.concat(
    [df, implicit_df],
    ignore_index=True
)

print(df.shape)

(48199, 3)


In [51]:
# leetspeak normalization

def normalize_leetspeak(text):

    leet_map = {

        '0': 'o',
        '1': 'i',
        '3': 'e',
        '4': 'a',
        '5': 's',
        '7': 't',
        '@': 'a',
        '$': 's'

    }

    for leet_char, normal_char in leet_map.items():

        text = text.replace(
            leet_char,
            normal_char
        )

    return text

In [52]:
# obfuscation handling
def fix_obfuscation(text):

    text = str(text)
    text = normalize_leetspeak(text)

    # Fix spaced characters like:
    # "f u c k" → "fuck"
    text = re.sub(
        r'\b(?:\w\s){2,}\w\b',
        lambda m: m.group(0).replace(" ", ""),
        text
    )

    # Generalized obfuscation patterns

    patterns = {

        r'd[\W_]*u[\W_]*m[\W_]*b': 'dumb',

        r'f[\W_]*u[\W_]*c[\W_]*k': 'fuck',

        r'b[\W_]*i[\W_]*t[\W_]*c[\W_]*h': 'bitch',

        r'i[\W_]*d[\W_]*i[\W_]*o[\W_]*t': 'idiot',

        r's[\W_]*t[\W_]*u[\W_]*p[\W_]*i[\W_]*d': 'stupid'

    }

    for pattern, replacement in patterns.items():

        text = re.sub(
            pattern,
            replacement,
            text,
            flags=re.IGNORECASE
        )

    # Normalize elongated words
    # "fuuuuuck" → "fuuck"

    text = re.sub(r'(.)\1{2,}', r'\1\1', text)

    return text

In [53]:
df["text_obf"] = df["clean_text"].apply(fix_obfuscation)

In [54]:
df[["clean_text", "text_obf"]].head(10)

,clean_text,text_obf
0,so if he hit her in the same way for what she ...,so if he hit her in the same way for what she ...
1,oh fuck im late i missed stupid shit im sure mkr,oh fuck im late i missed stupid shit im sure mkr
2,they get attitude like bitch too i think some ...,they get attitude like bitch too i think some ...
3,im mad james let me drink that shit payback is...,im mad james let me drink that shit payback is...
4,wait who respect colin mkr,wait who respect colin mkr
5,dis be one sexy bitch,dis be one sexy bitch
6,rt republican rick scott ha been reelected flg...,rt republican rick scott ha been reelected flg...
7,so true,so true
8,id be happy to debate the lie this islamist po...,id be happy to debate the lie this islamist po...
9,are you retarded you think iraq afghan were av...,are you retarded you think iraq afghan were av...


In [55]:
# test obfuscation

samples = [

    "du_mb",

    "du#mb",

    "d.u.m.b",

    "fu_ck",

    "b!tch",

    "1diot",

    "1d10t",

    "n1gger",

    "h4te",

    "f@ck",

    "stup1d"

]
for s in samples:

    print("ORIG:", s)

    print("FIX :", fix_obfuscation(s))

    print("-"*40)

ORIG: du_mb
FIX : dumb
----------------------------------------
ORIG: du#mb
FIX : dumb
----------------------------------------
ORIG: d.u.m.b
FIX : dumb
----------------------------------------
ORIG: fu_ck
FIX : fuck
----------------------------------------
ORIG: b!tch
FIX : b!tch
----------------------------------------
ORIG: 1diot
FIX : idiot
----------------------------------------
ORIG: 1d10t
FIX : idiot
----------------------------------------
ORIG: n1gger
FIX : nigger
----------------------------------------
ORIG: h4te
FIX : hate
----------------------------------------
ORIG: f@ck
FIX : fack
----------------------------------------
ORIG: stup1d
FIX : stupid
----------------------------------------


In [56]:
slang_dict={

     "u": "you",
    "ur": "your",
    "ya": "you",
    "yo": "you",
    "im": "i am",
    "dont": "do not",
    "cant": "cannot",
    "wont": "will not",
    "nigga": "nigger",
    "niggah": "nigger",
    "btch": "bitch",
    "fck": "fuck",
    "wtf": "what the fuck",
    "omg": "oh my god",
    "afaik": "as far as i know",
    "afk": "away from keyboard",
    "asl": "age sex location",
    "atm": "at the moment",
    "b/c": "because",
    "b/w": "between",
    "b4": "before",
    "bbiab": "be back in a bit",
    "bbl": "be back later",
    "bff": "best friends forever",
    "brb": "be right back",
    "btw": "by the way",
    "ctn": "cannot talk now",
    "cye": "check your email",
    "dl": "download",
    "eta": "estimated time of arrival",
    "fwiw": "for what it is worth",
    "fyi": "for your information",
    "gg": "good game",
    "gj": "good job",
    "gl": "good luck",
    "gr8": "great",
    "gtg": "got to go",
    "gmv": "got my vote",
    "hth": "hope this helps",
    "hw": "homework",
    "iac": "in any case",
    "ic": "i see",
    "idk": "i do not know",
    "iirc": "if i remember correctly",
    "ikr": "i know right",
    "im": "i am",
    "imo": "in my opinion",
    "imho": "in my humble opinion",
    "irl": "in real life",
    "jk": "just kidding",
    "k": "ok",
    "l8": "late",
    "l8r": "later",
    "lmk": "let me know",
    "lol": "laughing out loud",
    "mmb": "message me back",
    "msg": "message",
    "myob": "mind your own business",
    "n/a": "not available",
    "nc": "no comment",
    "ne1": "anyone",
    "nm": "not much",
    "noob": "newbie",
    "np": "no problem",
    "ntn": "no thanks needed",
    "omg": "oh my gosh",
    "omw": "on my way",
    "ot": "off topic",
    "pc": "personal computer",
    "pls": "please",
    "pos": "parent over shoulder",
    "ppl": "people",
    "qt": "cutie",
    "re": "regarding",
    "smh": "shaking my head",
    "sry": "sorry",
    "tba": "to be announced",
    "tbc": "to be continued",
    "tc": "take care",
    "thx": "thanks",
    "tia": "thanks in advance",
    "tlc": "tender loving care",
    "tmi": "too much information",
    "ttfn": "ta ta for now",
    "ttyl": "talk to you later",
    "txt": "text",
    "ty": "thank you",
    "w/e": "whatever",
    "w/o": "without",
    "w8": "wait",
    "xoxo": "hugs and kisses",
    "y": "why",
    "ynt": "why not",
    "yolo": "you only live once",
    "yw": "you are welcome",
    "zzz": "sleeping"
}

In [57]:
def normalize_slang(text):
    words = text.split()
    normalized = [slang_dict.get(w, w) for w in words]
    return " ".join(normalized)

In [58]:
df["text_slang"] = df["text_obf"].apply(normalize_slang)

In [59]:
df[["text_obf", "text_slang"]].head(10)

,text_obf,text_slang
0,so if he hit her in the same way for what she ...,so if he hit her in the same way for what she ...
1,oh fuck im late i missed stupid shit im sure mkr,oh fuck i am late i missed stupid shit i am su...
2,they get attitude like bitch too i think some ...,they get attitude like bitch too i think some ...
3,im mad james let me drink that shit payback is...,i am mad james let me drink that shit payback ...
4,wait who respect colin mkr,wait who respect colin mkr
5,dis be one sexy bitch,dis be one sexy bitch
6,rt republican rick scott ha been reelected flg...,rt republican rick scott ha been reelected flg...
7,so true,so true
8,id be happy to debate the lie this islamist po...,id be happy to debate the lie this islamist po...
9,are you retarded you think iraq afghan were av...,are you retarded you think iraq afghan were av...


In [60]:
df.sample(10)[["text_obf", "text_slang"]]

,text_obf,text_slang
16776,i wa gon na hold my tongue until i could get i...,i wa gon na hold my tongue until i could get i...
46197,rt when you thinking about killing a bitch,rt when you thinking about killing a bitch
8226,rt eatin the pussy right and she push you off ...,rt eatin the pussy right and she push you off ...
25426,of course you would think that being a liberal...,of course you would think that being a liberal...
42140,if the world wa free of sucka free it would be...,if the world wa free of sucka free it would be...
15922,it just not the same without manu mkr,it just not the same without manu mkr
34786,the jew once lived on ao of the arabian pennin...,the jew once lived on ao of the arabian pennin...
43220,when ur writing a book about wit it is prob no...,when your writing a book about wit it is prob ...
12845,i m not sure why we care about any of the wari...,i m not sure why we care about any of the wari...
21937,kobane won against daesh and their partner erd...,kobane won against daesh and their partner erd...


In [61]:
samples = [
    "u coon cant wait",
    "im gonna kill him",
    "wtf is this",
]

for s in samples:
    print("ORIG:", s)
    print("SLNG:", normalize_slang(s))
    print("-"*40)

ORIG: u coon cant wait
SLNG: you coon cannot wait
----------------------------------------
ORIG: im gonna kill him
SLNG: i am gonna kill him
----------------------------------------
ORIG: wtf is this
SLNG: what the fuck is this
----------------------------------------


In [62]:
hate_keywords = [
    "nigger", "coon", "bitch", "fuck", "feminazi",
    "whore", "slut", "rape", "kill", "terrorist",
    "scum", "slutty", "trollop", "prude",
    "man eater", "cougar",
    "mutton dressed as lamb",
    "faded beauty",
    "catty", "bitchy",
    "gold digger", "bimbo",
    "asking for it",
     "chick", "dame",
    "witch", "matronly","bigot", "bigotry", "nigga", "n-word","wop","paddy","dago","polack"
    
]

In [63]:
def keyword_flag(text):
    text = text.lower()
    return int(any(kw in text for kw in hate_keywords))

In [64]:
df["keyword_flag"] = df["text_slang"].apply(keyword_flag)

In [65]:
df.sample(10)[["text_slang", "keyword_flag"]]

,text_slang,keyword_flag
34234,you homegirl fat and got a mustache no need to...,1
30707,cam newton is such a little girl like grow up ...,1
41487,the markov bot are on a roll this morning,0
46666,whats with rob in the hat mate we know youre b...,0
38816,rt sucking dick amp eating pussy is apart of s...,1
28674,happy birthday queer,0
31417,were butcher but cannot tell red poultry is no...,0
35594,i am back bitch,1
5931,he used to be like ioost himself,0
46065,how wa that not a strike you buffoon,0


In [66]:
def mask_keywords(text, prob=0.3):
    words = text.split()
    masked = []

    for w in words:
        if w in hate_keywords and random.random() < prob:
            masked.append("[MASK]")
        else:
            masked.append(w)

    return " ".join(masked)

In [67]:
df["text_masked"] = df["text_slang"].apply(mask_keywords)

In [68]:
df.sample(10)[["text_slang", "text_masked"]]

,text_slang,text_masked
8196,i head butted a bitch in the club lmao she tri...,i head butted a [MASK] in the club lmao she tr...
7037,woman from my avatar should be on number i p,woman from my avatar should be on number i p
3164,822o i legit watched it for about eo or ao sec...,822o i legit watched it for about eo or ao sec...
29317,yall dumb bitch out here fucking up,yall dumb bitch out here fucking up
622,yeah male shepard is just so different and int...,yeah male shepard is just so different and int...
5159,u2oicthe idea that chappelle in whiteface is p...,u2oicthe idea that chappelle in whiteface is p...
29223,i am at home throwing up taking care of my kid...,i am at home throwing up taking care of my kid...
27451,for victim of abuse it often not discussed bec...,for victim of abuse it often not discussed bec...
41800,rt a bird doesnt sing because it ha an answer ...,rt a bird doesnt sing because it ha an answer ...
4139,march is about to get mad marchmadness college...,march is about to get mad marchmadness college...


In [69]:
test_samples = [
    "fuck you bitch",
    "you are a stupid bitch",
    "this is a normal sentence",
    "i will kill you",
    "she is a gold digger",
    "hello how are you",
    "nigger please",
    "you are amazing"
]

for text in test_samples:
    masked = mask_keywords(text, prob=1.0)  # force masking for testing
    
    print("ORIG :", text)
    print("MASK :", masked)
    print("-" * 50)

ORIG : fuck you bitch
MASK : [MASK] you [MASK]
--------------------------------------------------
ORIG : you are a stupid bitch
MASK : you are a stupid [MASK]
--------------------------------------------------
ORIG : this is a normal sentence
MASK : this is a normal sentence
--------------------------------------------------
ORIG : i will kill you
MASK : i will [MASK] you
--------------------------------------------------
ORIG : she is a gold digger
MASK : she is a gold digger
--------------------------------------------------
ORIG : hello how are you
MASK : hello how are you
--------------------------------------------------
ORIG : nigger please
MASK : [MASK] please
--------------------------------------------------
ORIG : you are amazing
MASK : you are amazing
--------------------------------------------------


In [70]:
df["final_text"] = df["text_masked"]

In [71]:
df_final = df[["final_text", "class", "keyword_flag"]]

In [72]:
# =====================================
# SAVE FINAL IMPROVED DATASET
# =====================================

print("Final dataset shape:", df.shape)

print("\nClass distribution:")
print(df["class"].value_counts())

df.to_csv(
    r"C:\Users\hp\Downloads\UNI\Spring 2026\Bachelor\Bachelor_Project\data\improved_dataset_final.csv",
    index=False
)

print("\nDataset saved successfully.")

Final dataset shape: (48199, 8)

Class distribution:
class
1    27911
0    20288
Name: count, dtype: int64

Dataset saved successfully.


In [73]:
print(df_final.shape)
df_final.head()

(48199, 3)


,final_text,class,keyword_flag
0,so if he hit her in the same way for what she ...,1,0
1,oh [MASK] i am late i missed stupid shit i am ...,0,1
2,they get attitude like [MASK] too i think some...,1,1
3,i am mad james let me drink that shit payback ...,1,1
4,wait who respect colin mkr,0,0


In [74]:
df_final[df_final["final_text"].str.strip() == ""]

,final_text,class,keyword_flag


In [75]:
df_final.sample(20)

,final_text,class,keyword_flag
18827,following the example of the pedophile prophet...,1,0
17959,hahahahahaha oh god i cannot even mkr promolife,0,0
46579,maybe you didnt notice that wa an august artic...,0,0
9041,weird,0,0
4588,nope it directed at non believer,0,0
26489,yep not sexist at all,0,0
22852,only broke bitch steal money,1,1
10172,oh shit i28ssti28i28 rt 822o dallas trash822i ...,1,0
6144,karma karma karma mkr,0,0
21126,laughing out loud bitch from riverside to la ahah,1,1


In [76]:
df_final["keyword_flag"].value_counts()

keyword_flag
0    31879
1    16320
Name: count, dtype: int64